In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# 1. Laad en bereid de dataset voor
df = pd.read_csv('cedint_f1_model_ready.csv')
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'])

def calculate_ashrae_metrics(y_true, y_pred, split_name, zone, load):
    if len(y_true) == 0: return None
    
    n = len(y_true)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mean_y = np.mean(y_true)
    
    # ASHRAE 14 Formules
    cv_rmse = (rmse / mean_y) * 100 if mean_y != 0 else 0
    nmbe = (np.sum(y_true - y_pred) / (n * mean_y)) * 100 if mean_y != 0 else 0
    
    # Strenge limieten: CV(RMSE) <= 30% en NMBE tussen -10% en 10%
    ashrae_pass = (cv_rmse <= 30) and (-10 <= nmbe <= 10)
    
    return {
        'zone': zone, 'load': load, 'model': 'gbm', 'split': split_name,
        'n': n, 'MAE': mae, 'RMSE': rmse, 'CV_RMSE': cv_rmse, 'NMBE': nmbe, 
        'SHRAE_pass': ashrae_pass
    }

# 2. Definieer de zones en load types waar we over willen loopen
zones = ['NW', 'NE', 'SE', 'SW']
load_types = ['HVAC', 'Lighting', 'Plugs & Sockets']

all_metrics = []
all_predictions = pd.DataFrame()

# 3. Start de loop over alle combinaties in het gebouw
for zone in zones:
    for load in load_types:
        # Filter de data voor deze specifieke zone en sensor
        df_subset = df[(df['zone'] == zone) & (df['load_type'] == load)].copy()
        
        # Sla over als de combinatie niet bestaat (bijv. geen HVAC in zone NE/SW als dat zo is)
        if df_subset.empty:
            continue
            
        print(f"Training GBM Model voor Zone: {zone} - Sensor: {load}...")
        df_subset = df_subset.sort_values('timestamp_local')
        
        # Features berekenen voor deze specifieke subset
        df_subset['energy_lag_24'] = df_subset['energy_wh_model'].shift(24)
        df_subset['energy_lag_168'] = df_subset['energy_wh_model'].shift(168)
        df_subset['temp_lag1'] = df_subset['temp_c'].shift(1)
        df_subset['temp_lag2'] = df_subset['temp_c'].shift(2)
        df_subset['roll_mean_24'] = df_subset['energy_wh_model'].rolling(window=24).mean()
        df_subset['dh_heat'] = (18 - df_subset['temp_c']).clip(lower=0)
        df_subset['dh_cool'] = (df_subset['temp_c'] - 24).clip(lower=0)
        
        df_subset = df_subset.dropna(subset=['energy_wh_model', 'energy_lag_168', 'roll_mean_24'])
        
        features = [
            "hour_local", "dayofweek_local", "month_local", "is_weekend", "is_holiday", 
            "temp_c", "dh_cool", "dh_heat", "temp_lag1", "temp_lag2",                    
            "occupancy", "lum_lux",                                                      
            "energy_lag_24", "energy_lag_168", "roll_mean_24"                            
        ]
        
        X = df_subset[features].fillna(0)
        y = df_subset['energy_wh_model']
        
        test_mask = df_subset['include_eval'] == 1
        X_test = X[test_mask]
        y_test = y[test_mask]
        
        train_mask_with_covid = df_subset['include_train'] == 1
        X_train_with = X[train_mask_with_covid]
        y_train_with = y[train_mask_with_covid]
        
        if len(X_train_with) == 0 or len(X_test) == 0:
            continue
            
        # Model trainen
        gbm_with = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
        gbm_with.fit(X_train_with, y_train_with)
        preds_with = gbm_with.predict(X_test)
        
        # Voorspellingen toevoegen aan de grote verzamel-lijst
        output_df = pd.DataFrame({
            'timestamp_utc': df_subset.loc[test_mask, 'timestamp_local'],
            'zone': zone,
            'load_type': load,
            'model': 'gbm',
            'split': 'test',
            'y_true': y_test,      
            'y_pred': preds_with   
        })
        all_predictions = pd.concat([all_predictions, output_df])
        
        # Test sets voor 2023 en 2025 filteren voor Metrics en Grafieken
        mask_2023 = test_mask & (df_subset['timestamp_local'].dt.year == 2023)
        mask_2025 = test_mask & (df_subset['timestamp_local'].dt.year == 2025)
        
        X_2023, y_2023 = X[mask_2023], y[mask_2023]
        X_2025, y_2025 = X[mask_2025], y[mask_2025]
        
        preds_2023 = gbm_with.predict(X_2023) if len(X_2023) > 0 else []
        preds_2025 = gbm_with.predict(X_2025) if len(X_2025) > 0 else []
        
        # Metrics verzamelen
        if len(y_2023) > 0: all_metrics.append(calculate_ashrae_metrics(y_2023, preds_2023, 'test_2023', zone, load))
        if len(y_2025) > 0: all_metrics.append(calculate_ashrae_metrics(y_2025, preds_2025, 'test_2025', zone, load))
        
        # Grafiek genereren
        fig, axes = plt.subplots(2, 1, figsize=(14, 10), facecolor='#FAFAF0')
        
        # Grafiek 2023
        if len(y_2023) > 0:
            axes[0].plot(df_subset.loc[X_2023.index, 'timestamp_local'], y_2023, color='#555555', label='Valore Reale', linewidth=1.5)
            axes[0].plot(df_subset.loc[X_2023.index, 'timestamp_local'], preds_2023, color='#4A6BB2', label='Predizione', linewidth=1)
        axes[0].set_title(f'Test 2023 - {load} Zona {zone}', fontweight='bold')
        axes[0].set_ylabel('Consumo (Wh)')
        axes[0].grid(color='#E5D8C1', linestyle='--', linewidth=0.5)
        axes[0].legend()
        axes[0].set_facecolor('#FFFEF8')
        
        # Grafiek 2025
        if len(y_2025) > 0:
            axes[1].plot(df_subset.loc[X_2025.index, 'timestamp_local'], y_2025, color='#555555', label='Valore Reale', linewidth=1.5)
            axes[1].plot(df_subset.loc[X_2025.index, 'timestamp_local'], preds_2025, color='#4A6BB2', label='Predizione', linewidth=1)
        axes[1].set_title(f'Test 2025 - {load} Zona {zone}', fontweight='bold')
        axes[1].set_ylabel('Consumo (Wh)')
        axes[1].set_xlabel('Timestamp Locale')
        axes[1].grid(color='#E5D8C1', linestyle='--', linewidth=0.5)
        axes[1].legend()
        axes[1].set_facecolor('#FFFEF8')
        
        plt.tight_layout()
        safe_load_name = load.replace(' ', '_').replace('&', 'and')
        plt.savefig(f'{zone}_{safe_load_name}_GBM_Grafiek.png', dpi=300)
        plt.close() # Zorgt dat we met een leeg canvas beginnen aan de volgende loop

# 4. Alles opslaan (BUITEN de loop)
all_predictions.to_csv('All_Zones_GBM_predictions.csv', index=False)

# Filter 'None' waardes uit de metrics lijst voor we opslaan
df_metrics = pd.DataFrame([m for m in all_metrics if m is not None])
df_metrics.to_excel('All_Zones_GBM_ASHRAE_metrics.xlsx', index=False)

print("\n=======================================================")
print("SUCCES! Alle GBM-modellen zijn getraind en geëxporteerd.")
print("Je hebt nu één master CSV en Excel bestand voor de Blend!")
print("=======================================================")

Training GBM Model voor Zone: NW - Sensor: HVAC...
Training GBM Model voor Zone: NW - Sensor: Lighting...
Training GBM Model voor Zone: NW - Sensor: Plugs & Sockets...
Training GBM Model voor Zone: NE - Sensor: HVAC...
Training GBM Model voor Zone: NE - Sensor: Plugs & Sockets...
Training GBM Model voor Zone: SE - Sensor: HVAC...
Training GBM Model voor Zone: SE - Sensor: Lighting...
Training GBM Model voor Zone: SE - Sensor: Plugs & Sockets...
Training GBM Model voor Zone: SW - Sensor: Lighting...


C:\Users\joosv\AppData\Local\Temp\ipykernel_29968\516884160.py:136: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[1].legend()


Training GBM Model voor Zone: SW - Sensor: Plugs & Sockets...


C:\Users\joosv\AppData\Local\Temp\ipykernel_29968\516884160.py:136: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[1].legend()



SUCCES! Alle GBM-modellen zijn getraind en geëxporteerd.
Je hebt nu één master CSV en Excel bestand voor de Blend!


In [2]:
import os
print(os.getcwd())

c:\Users\joosv\Downloads
